In [1]:
# --- system setup ---
import sys
import os
sys.path.append(os.path.abspath(".."))

In [2]:
from datetime import date
import pandas as pd

In [3]:
from ib_insync import *
from ibkr.Class_IBKR_IB import IBKR_IB
ibkr = IBKR_IB(port=7496)

async def start_ibkr():
    await ibkr.connect()
    print("IBKR connected:", ibkr.ib.isConnected())

await start_ibkr()

IBKR connected: True


In [4]:
prices_to_use = "TRADES"

'''
    "TRADES"
    "MIDPOINT"
    "BID"
    "ASK"
    "BID_ASK"
    "ADJUSTED_LAST"
    "HISTORICAL_VOLATILITY"
    "OPTION_IMPLIED_VOLATILITY"
    "FEE_RATE"
    "REBATE_RATE"
    "SCHEDULE"      
'''

'\n    "TRADES"\n    "MIDPOINT"\n    "BID"\n    "ASK"\n    "BID_ASK"\n    "ADJUSTED_LAST"\n    "HISTORICAL_VOLATILITY"\n    "OPTION_IMPLIED_VOLATILITY"\n    "FEE_RATE"\n    "REBATE_RATE"\n    "SCHEDULE"      \n'

In [5]:
'''
lookback_period = "2 Y"
length_of_each_period = "1 day"
use_regular_trading_hours = True
'''

#'''
lookback_period = "5 D"
length_of_each_period = "1 day"
use_regular_trading_hours = True
#'''

In [7]:
'''
symbols = ['ARKB', 
           'BITB', 
           'BRRR', 
           'BTC', 
           'BTCW', 
           'EZBC', 
           'FBTC',
           'GBTC', 
           'HODL', 
           'IBIT']  
'''

'''
symbols = ['PCY',
           'EMLC',
           'FALN',
           'ANGL',
           'PGX',
           'PFFD',
           'PFF',
           'BWX',
           'IGOV',
           'CWB',
           'ICVT',
           'BNDX',
           'IAGG',
           'SPBO',
           'USIG',
           'VTC',
           'CORP',
           'LQD',
           'SPTL',
           'SCHQ',
           'VGLT',
           'TLT',
           'CMF',
           'VTEC',
           'TFI',
           'VTEB',
           'MUB',
           'SPHY',
           'SCYB',
           'HYLB',
           'USHY',
           'HYG',
           'JNK',
           'SPMB',
           'VMBS',
           'MBB',
           'VWOB',
           'EMB',
           'SCHZ',
           'SPAB',
           'IUSB',
           'BND',
           'AGG',
           'SPLB',
           'IGLB',
           'VCLT']
'''             

#'''
symbols = ['SPLB', 'IGLB', 'VCLT']
           
#'''

'''
conIds = [320106059, 641561653]
'''

'\nconIds = [320106059, 641561653]\n'

In [8]:
df_list = []

for sym in symbols:
#for conId in conIds:

    contract = Stock(sym, 'SMART', 'USD')
    await ibkr.ib.qualifyContractsAsync(contract)

    # contract = await ibkr.contract_by_conId(conId)
    # contract = Future(symbol='BRR', lastTradeDateOrContractMonth='202606', exchange='CME', currency='USD')

    bars = await ibkr.ib.reqHistoricalDataAsync(
        contract=contract,
        endDateTime="",          # "" means now
        durationStr=lookback_period,
        barSizeSetting=length_of_each_period,
        whatToShow=prices_to_use,
        useRTH=use_regular_trading_hours,
        formatDate=1
    )

    df = util.df(bars)

    '''
    df['date'] = pd.to_datetime(df['date'])
    df["time"] = df["date"].dt.time
    df['date'] = df['date'].dt.date
    '''

    '''
    df.to_csv("historical_prices_" + contract.localSymbol + ".csv", index=False)
    df
    '''

    df[sym] = df['close']
    df = df.set_index("date")
    df_list.append(df[sym])

big_df = pd.concat(df_list, axis=1)

today_text = date.today().strftime("%Y-%m-%d")
big_df.to_csv("stat_arb_prices_" + today_text + ".csv", index=True)


